# Playground — NLP Sentiment Models

Cada sección entrena (o carga desde checkpoint) un modelo. Al final hay una celda unificada para comparar predicciones.

In [ ]:
import sys, importlib.util
from pathlib import Path

ROOT = Path().resolve()  # class_pip/
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def _load(name, subdir, file="pipeline.py"):
    """Carga un módulo de un subdirectorio sin conflictos de nombre."""
    subdir_path = ROOT / subdir
    if str(subdir_path) not in sys.path:
        sys.path.insert(0, str(subdir_path))
    spec = importlib.util.spec_from_file_location(name, subdir_path / file)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

models = {}  # se puebla a medida que se cargan modelos

---
## 1. VADER (lexicon, sin entrenamiento)

In [ ]:
lexicon = _load("lexicon_pipeline", "lexicon")
lexicon.main()

---
## 2. TF-IDF + Naive Bayes

In [ ]:
nb = _load("nb_pipeline", "naivebayes")
nb.main()

In [ ]:
models["Naive Bayes"] = nb.predict

---
## 3. TF-IDF + Logistic Regression

In [ ]:
lr = _load("lr_pipeline", "logistic")
lr.main()

In [ ]:
models["Logistic"] = lr.predict

---
## 4. BiLSTM (embeddings aleatorios)

In [ ]:
sent = _load("sent_pipeline", "sentiment")
sent.main()

In [ ]:
sent_pred = _load("sent_predict", "sentiment", file="predict.py")
models["BiLSTM"] = sent_pred.predict

---
## 5. BiLSTM + GloVe

In [ ]:
glove = _load("glove_pipeline", "embeddings")
glove.main()

In [ ]:
glove_pred = _load("glove_predict", "embeddings", file="predict.py")
models["BiLSTM+GloVe"] = glove_pred.predict

---
## 6. DistilBERT (fine-tuning)

In [ ]:
bert = _load("bert_pipeline", "finetune")
bert.main()

In [ ]:
bert_pred = _load("bert_predict", "finetune", file="predict.py")
models["DistilBERT"] = bert_pred.predict

---
## Playground — predicción unificada

Cambiá `tweet` y ejecutá la celda.

In [ ]:
tweet = "I can't believe how bad this is"

print(f"Tweet: {tweet!r}\n")
print(f"{'Model':<16} {'Label':<10} {'Confidence'}")
print("-" * 38)
for name, predict_fn in models.items():
    label, prob = predict_fn(tweet)
    print(f"{name:<16} {label:<10} {prob:.1%}")